# READ ME FIRST (DELETE THIS CELL BEFORE SUBMITTING)

Every **markdown** cell in this notebook is written as final, submission-ready text. Every **code** cell holds numbered `#` instructions for the code to write; keep them as one-line comments under the code you write.

**Two kinds of gaps remain, and both must be filled before submitting:**

* `[[FILL: ...]]` = a number, country name or short finding that only exists once the code has run. Copy it from the printed output above the cell. Never type a value that no output shows.
* The few interpretive sentences after a figure are written as templates with `[[FILL]]` slots, not as findings. If the output contradicts a sentence (for example, the template says "mostly event mix" but the table shows "mostly within"), rewrite the sentence to match the output.

Before exporting, search the notebook for `[[` (Ctrl+F). The count must be **0**.

**Verified on jojie (15 Nov 2026):** 2,335 files `YYYYMMDD.export.CSV.zip` (1 Apr 2013 to 26 Aug 2019), 24G by `du`, 4 missing days (2014-01-23/24/25, 2014-03-19), 58 tab-separated columns, `wc -l` of 20130401 = 27,758, 20160615 = 240,023, 20190826 = 157,071; 32 cores, 251 GB RAM, PySpark 4.1.1, plotly 6.6, kaleido 1.2, pandas 2.3.

**Rules for the code (Code Hygiene):** PEP 8 (79-character lines, `snake_case`, `UPPER_CASE` constants); numpy-style docstring on every function; one-line comments saying why; `_df` = Spark, `_pd` = pandas; relative paths only; never display the `spark` object or a raw `explain()`/traceback (they reveal the username); restart and Run All before exporting.

**Supplementary files to create:** `gdelt_io.py` (reader), `gdelt_lookups.py` (CAMEO, QuadClass, geo types, FIPS-to-ISO3, `SEA_FIPS`, `PH_EVENTS`), `gdelt_analysis.py` (statistics), `gdelt_plots.py` (one function per figure). Their contents are listed in Table 5 and in the code cells that use them. `FIPS_TO_ISO3` must use FIPS 10-4 codes: AS = Australia, AU = Austria, RP = Philippines, CH = China, GM = Germany, JA = Japan, KS = South Korea, RS = Russia, SP = Spain, IZ = Iraq, NI = Nigeria, NG = Niger, SW = Sweden, SZ = Switzerland, UK = United Kingdom, BM = Myanmar, CB = Cambodia, VM = Vietnam, SN = Singapore, BX = Brunei, TT = Timor-Leste.

# The World's Mood Ring: How the Tone of Global News Varies by Country and Season

BDCC 3301 Big Data and Cloud Computing · Lab 2 Technical Report

## Abstract

How a country is portrayed in global news shapes how the rest of the world sees it, yet that portrayal is rarely measured for all countries at once. The GDELT 1.0 Event Database records, for every news event it detects, where the event took place and the average tone of the articles that reported it. This study asks: **which countries does global news portray most positively and most negatively, and does that tone follow seasonal patterns?** Apache Spark was used to parse all 2,335 daily GDELT event files published between 1 April 2013 and 26 August 2019 (24 GB compressed), yielding [[FILL: N_CLEAN]] located events in [[FILL: number of eligible countries]] countries. Tone was aggregated by country, month of publication and CAMEO event category. Countries were ranked with month-bootstrap confidence intervals, seasonal indices were tested with a permutation test under false-discovery control, and each country's gap from the global mean was decomposed into the kind of events that happen there and the way those events are reported. [[FILL: three most positive countries]] received the most positive coverage (mean tone [[FILL]] to [[FILL]]), and [[FILL: three most negative countries]] the most negative ([[FILL]] to [[FILL]]), against a global mean of [[FILL]]. Tone showed significant seasonality in [[FILL: k]] of [[FILL: n]] countries, with a median amplitude of [[FILL]] tone points, [[FILL: smaller / larger]] than the typical gap between countries. For the most negatively covered countries, [[FILL: event mix / within-category tone]] explained most of the gap. The Philippines ranked [[FILL]] of [[FILL]] countries and [[FILL: one-sentence Philippines finding]]. Tourism and communications offices can use the same pipeline as a monthly benchmark of national image, focusing on [[FILL: the event categories or months the results point to]].

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Data Source and Description](#2.-Data-Source-and-Description)
3. [Methodology](#3.-Methodology)
4. [Environment and Data Ingestion](#4.-Environment-and-Data-Ingestion)
5. [Exploratory Data Analysis and Null Investigation](#5.-Exploratory-Data-Analysis-and-Null-Investigation)
6. [Data Filtering](#6.-Data-Filtering)
7. [RQ1: Which countries are covered most positively and most negatively?](#7.-RQ1:-Which-countries-are-covered-most-positively-and-most-negatively?)
8. [RQ2: Does tone follow seasonal patterns?](#8.-RQ2:-Does-tone-follow-seasonal-patterns?)
9. [RQ3: Which event types drive each country's tone?](#9.-RQ3:-Which-event-types-drive-each-country's-tone?)
10. [RQ4: How is the Philippines covered compared with its Southeast Asian neighbours?](#10.-RQ4:-How-is-the-Philippines-covered-compared-with-its-Southeast-Asian-neighbours?)
11. [Results Summary and Conclusion](#11.-Results-Summary-and-Conclusion)
12. [Limitations, Recommendations and Future Work](#12.-Limitations,-Recommendations-and-Future-Work)
13. [References](#References)

Steps 1 to 30 are numbered across the notebook, and each step names the supplementary file it uses.

## 1. Introduction

### 1.1 Background

What people abroad know about a country comes largely from the news. Journalists and audiences pay more attention to negative than to positive information (Soroka & McAdams, 2015), so a country that appears mostly through crises, conflict or disasters can carry a darker image than its everyday reality. That image matters: a nation's reputation affects whether people visit, invest, study or trade there (Anholt, 2007).

Until recently, measuring that image meant reading samples of articles by hand. The Global Database of Events, Language, and Tone (GDELT) changes this. It monitors broadcast, print and online news from around the world in more than 100 languages, codes each event it finds with the CAMEO event taxonomy, places it on a map, and records the average tone of the articles that reported it (Leetaru & Schrodt, 2013). Every event therefore carries a measurement of how it was portrayed, which makes the portrayal of every country measurable at scale.

### 1.2 Problem Statement

**Which countries does global news portray most positively and most negatively, and does that tone follow seasonal patterns?**

Here, tone means GDELT's `AvgTone`: the average tone of all articles that mentioned an event, on a scale from -100 (extremely negative) to +100 (extremely positive), with most values between -10 and +10 (GDELT Project, 2013).

### 1.3 Motivation

Solving this problem matters because a country's news image is an asset that many institutions try to manage but few can measure. Three groups would act on the answer:

1. **Tourism boards and investment promotion agencies** spend on image campaigns, but they rarely know where their country stands against others before the campaign starts. A ranking of every country by news tone gives them that baseline, and a monthly series lets them track it afterwards.
2. **Government communications offices** need to know *when* coverage turns darker and *why*. If tone follows the calendar, announcements and crisis preparation can be timed around it; if it is driven by particular kinds of events, the office knows which stories to prepare for.
3. **Researchers of media and international relations** usually study one outlet or one country at a time. A consistent measurement of millions of events across more than six years shows patterns that small samples cannot.

This study provides all three: a ranking and world map of news tone (RQ1), seasonal tone curves with a test of whether they exceed chance (RQ2), a breakdown of each country's tone into *what happens there* and *how it is reported* (RQ3), and a deep-dive on the Philippines and its Southeast Asian neighbours (RQ4).

### 1.4 Research Questions

The problem statement is answered through four research questions.

| RQ | Question | Section | Main output |
|---|---|---|---|
| RQ1 | Which countries have the most positive and most negative average news tone, and is that ranking robust? | 7 | Figures 9 to 11, Tables 26 to 27 |
| RQ2 | Does tone follow a calendar-month pattern, globally and per country, beyond what chance would produce? | 8 | Figures 12 to 15, Tables 28 to 29 |
| RQ3 | Which event types drive each country's tone: what happens there, or how it is reported? | 9 | Figures 16 to 17, Table 30 |
| RQ4 | How does coverage of the Philippines compare with its Southeast Asian neighbours in level, trend, season and drivers? | 10 | Figures 18 to 21, Tables 31 to 33 |

RQ1 and RQ2 answer the problem statement directly; RQ3 explains the answer; RQ4 applies the full analysis to the Philippines, the deep-dive promised in the study proposal.

## 2. Data Source and Description

### 2.1 Source and Collection

**What the data are.** The GDELT 1.0 Event Database is an open record of events reported in world news, covering 1979 to the present and containing over a quarter of a billion records (Leetaru & Schrodt, 2013). Each record is one event: who did what to whom, where, when, and how the news described it.

**How the source collected them.** GDELT's collection works in five stages (Leetaru & Schrodt, 2013; GDELT Project, 2013):

1. GDELT continuously monitors news from thousands of broadcast, print and online sources worldwide.
2. Software reads each article and extracts events as *actor - action - target*, coded with CAMEO (Conflict and Mediation Event Observations), a taxonomy of about 300 event types grouped into 20 root categories (Schrodt, 2012).
3. Each event is geocoded to the most precise place the article names (a country, a first-level region or a city), using FIPS 10-4 country codes.
4. Mentions of the same event are merged, and each event record stores how many mentions, sources and articles reported it, together with `AvgTone`, the average tone of those articles.
5. Since 1 April 2013, GDELT publishes one file per day, named `YYYYMMDD.export.CSV.zip`, holding the events added to the database that day. Before that date, records were published in monthly and yearly files organised by the date the event took place (GDELT Project, n.d.).

**Where the data are read from.** The files are read in place from the shared folder `/mnt/data/public/gdeltv1/events` on Jojie and are never copied.

**What is used.** All 2,335 daily files from 1 April 2013 to 26 August 2019, the last day available on Jojie. Their total size is 24 GB (output of `du -sch` in `du_screenshot.png`). Four days have no file: 23, 24 and 25 January 2014 and 19 March 2014.

**What is not used, and why.** The same folder holds 114 yearly and monthly backfile archives (1979 to March 2013). They use an older 57-column format without `DATEADDED`, so the month of publication, which this study needs, cannot be measured. The `GDELT.MASTERREDUCEDV2` files are a reduced, one-event-per-day summary of the same data, and the separate `gkg` folder holds a different database (the Global Knowledge Graph). None of these are read.

### 2.2 Fields Used

Each daily file is tab-separated, has no header row, and has 58 columns. Table 1 lists the 19 columns this study reads.

**Table 1. The 19 raw GDELT 1.0 fields used in this study, their position in the 58-column row, their type, meaning (GDELT Project, 2013) and role in the analysis.**

| Field | Index | Type | Meaning | Role |
|---|---|---|---|---|
| `GLOBALEVENTID` | 0 | integer | Unique identifier of the event | Key |
| `SQLDATE` | 1 | integer (YYYYMMDD) | Date on which the event took place, as reported | Time of event (exploration only) |
| `IsRootEvent` | 25 | 0 / 1 | 1 if the event was found in the lead paragraph of the article | Robustness check |
| `EventCode` | 26 | text | Full CAMEO action code (e.g. `043`) | Event type |
| `EventBaseCode` | 27 | text | Second-level CAMEO code | Event type (kept, not analysed) |
| `EventRootCode` | 28 | text | CAMEO root category, `01` to `20` | Event type |
| `QuadClass` | 29 | 1 to 4 | Verbal cooperation, material cooperation, verbal conflict, material conflict | Event type |
| `GoldsteinScale` | 30 | -10 to +10 | Theoretical impact of the event type on a country's stability | Exploration only |
| `NumMentions` | 31 | integer | Mentions of the event across all source documents | Exploration only |
| `NumSources` | 32 | integer | Number of information sources that reported it | Exploration only |
| `NumArticles` | 33 | integer | Number of source documents that reported it | Weight (Eq. 2) |
| `AvgTone` | 34 | -100 to +100 | Average tone of all documents that mentioned the event | **Outcome** |
| `ActionGeo_Type` | 49 | 0 to 5 | Precision of the event location (0 = none, 1 = country, 2 = US state, 3 = US city, 4 = world city, 5 = world state) | Location |
| `ActionGeo_FullName` | 50 | text | Name of the location | Location (Table 33) |
| `ActionGeo_CountryCode` | 51 | FIPS 10-4 | Country where the event took place | **Location (country)** |
| `ActionGeo_ADM1Code` | 52 | text | First-level administrative region | Location (Table 33) |
| `ActionGeo_Lat` | 53 | decimal | Latitude of the location | Location (checks) |
| `ActionGeo_Long` | 54 | decimal | Longitude of the location | Location (checks) |
| `DATEADDED` | 56 | integer (YYYYMMDD) | Date the event was added to the database | **Time of publication** |

The other 39 columns describe the two actors (codes, names, countries, groups, religions, types and locations), the fractional date and the source URL. They are profiled for empty values in Step 11 but not analysed, because this study is about the place of an event, not its actors.

### 2.3 Why the Event Location and the Date Added

Three choices about the raw fields shape every result, so they are stated before the analysis.

1. **The country of an event is `ActionGeo_CountryCode`, the place where it happened**, not `Actor1CountryCode`, the country of the actor. The question is how a *place* is portrayed. Actor fields are also often empty or describe groups rather than countries: the first record of 1 April 2013 has the actor "THE INTERNATIONAL COMMUNITY" (Table 8).
2. **The month of an event is taken from `DATEADDED`, the day the article was processed**, not `SQLDATE`, the day the event took place. Articles often mention past events: the 1 April 2013 file contains an event dated 4 April 2003. Since this study measures coverage *by month of publication*, `DATEADDED` is the correct clock. Step 13 measures how often the two dates differ.
3. **Country codes are FIPS 10-4, not ISO 3166.** In FIPS, Australia is `AS` (in ISO, `AS` is American Samoa) and Austria is `AU`; the Philippines is `RP`, not `PH`. Every country name and map location is therefore obtained through a FIPS-to-ISO3 lookup table (Section 2.4).

### 2.4 Lookup Tables Built for This Study

Jojie provides no code tables for GDELT, so four small lookup tables were transcribed from the official documentation into `gdelt_lookups.py` (Table 2).

**Table 2. Lookup tables in `gdelt_lookups.py`, the documents they were transcribed from, their size and how each was checked.**

| Lookup | Source document and how it was produced | Entries | Check |
|---|---|---|---|
| CAMEO root categories | CAMEO codebook (Schrodt, 2012): names of the 20 root codes | 20 | All root codes found in the data are named (Table 15) |
| QuadClass names | GDELT codebook (GDELT Project, 2013) | 4 | All values found are named (Table 14) |
| Location precision (`ActionGeo_Type`) | GDELT codebook (GDELT Project, 2013) | 6 | All values found are named (Table 16) |
| FIPS 10-4 to ISO3 and country name | FIPS 10-4 country codes used by the GDELT geocoder (GDELT Project, 2013), matched by hand to ISO 3166-1 alpha-3 codes for mapping | [[FILL: number of entries]] | Every code in the data is either mapped or listed as unmapped (Table 22) |
| Southeast Asia | The ten ASEAN member states plus Timor-Leste | 11 | Used only in Section 10 |

## 3. Methodology

### 3.1 Methodology Overview

The study follows the eight stages of Figure 1, from 2,335 raw daily files to the answers to four research questions.

1. **Ingest.** All 2,335 daily zip files are read in parallel with Spark, 19 fields are kept, and the result is cached as Parquet (Steps 4 to 7).
2. **Explore.** Volume over time, the distribution of tone, event categories, empty values in all 58 columns, the lag between event and publication, and duplicate identifiers are described (Steps 8 to 15).
3. **Filter.** Malformed rows, invalid tone, duplicates, events without a location, codes that are not countries, and countries with too few events are removed, each step counted (Steps 16 to 17).
4. **Aggregate.** Two Spark aggregations build the tables every later step reads: tone sums by country and month of publication, and by country and CAMEO root category (Steps 18 and 26).
5. **RQ1: Rank.** Countries are ranked by mean tone with month-bootstrap confidence intervals, mapped, and re-ranked under four alternative definitions (Steps 19 to 21).
6. **RQ2: Seasonality.** Each country's tone relative to the world is detrended, its seasonal index is computed, and the index is tested against chance with a permutation test and false-discovery control (Steps 22 to 25).
7. **RQ3: Drivers.** Each country's gap from the global mean tone is split into an event-mix part and a within-category part (Steps 26 to 27).
8. **RQ4: The Philippines.** Stages 5 to 7 are applied to the 11 Southeast Asian countries, and the Philippines' tone is followed over time (Steps 28 to 30).

In [ ]:
# STEP 0 / FIGURE 1: pipeline diagram (uses plot_pipeline_diagram from gdelt_plots.py)

# 1. Import plot_pipeline_diagram and save_fig from gdelt_plots (this cell runs before Step 1, so import here).
# 2. In gdelt_plots.py, plot_pipeline_diagram() draws with matplotlib only (FancyBboxPatch + annotate arrows):
#    - 8 boxes, left-to-right in two rows, labelled exactly as the 8 stages in 3.1, with the step numbers under each label
#    - colour by stage type: blue = ingest/explore/filter, green = aggregate, orange = the four RQ boxes
#    - a side box "Raw: 2,335 zips, 24 GB" at the start, and a box "Report: Sections 11-12" at the end
#    - the four RQ boxes all branch from the "Aggregate" box (arrows), they are not a chain
#    - a small legend for the three colours
# 3. Call it, save as figures/fig01_pipeline.png with save_fig, and show it.
# 4. No title inside the figure (the caption carries it).

**Figure 1. Pipeline of the study, from the 2,335 raw GDELT daily files to the four research questions. Blue boxes prepare the data, the green box builds the two aggregate tables that every analysis reads, and orange boxes answer RQ1 to RQ4. Every number in the report traces back to one Spark pass over the raw files (Step 6).**

### 3.2 Key Concepts and Equations

**Tone.** GDELT scores each article's tone with a sentiment dictionary, from the share of positive and negative words it contains; an event's `AvgTone` is the average over all articles that mentioned it (Leetaru & Schrodt, 2013; GDELT Project, 2013). A value below zero means the coverage used more negative than positive language.

**Pooled mean tone.** The main measure of how a country is portrayed is the mean tone of all located events in that country:

$$\bar{T}_c = \frac{1}{N_c}\sum_{i \in c} \text{AvgTone}_i \qquad \text{(Eq. 1)}$$

where $N_c$ is the number of events located in country $c$. The global mean $\bar{T}_g$ is the same average over all located events.

**Article-weighted mean.** An event reported by many articles may matter more to a country's image than one reported once. As a robustness check, each event is weighted by its number of articles $w_i$:

$$\bar{T}^{\,art}_c = \frac{\sum_{i\in c} w_i\,\text{AvgTone}_i}{\sum_{i\in c} w_i} \qquad \text{(Eq. 2)}$$

**Month-block bootstrap.** Events published in the same month share news cycles, so they are not independent, and an event-level standard error would make intervals too narrow. Confidence intervals are therefore computed by resampling the 77 months with replacement, recomputing Eq. 1 from the resampled monthly sums, and taking the 2.5th and 97.5th percentiles of 2,000 resamples (Efron & Tibshirani, 1993).

**Relative tone.** With $T_{c,m}$ the mean tone of country $c$ in publication month $m$ and $G_m$ the global mean tone in that month,

$$R_{c,m} = T_{c,m} - G_m \qquad \text{(Eq. 3)}$$

measures whether a country was covered better or worse than the world in that month. It removes changes that affect all countries at once, such as changes in the sources GDELT monitors (Wang et al., 2016).

**Trend and anomaly.** The slow trend of $R$ is estimated with a centred 2x12 moving average (Hyndman & Athanasopoulos, 2021):

$$\hat{\tau}_{c,m} = \tfrac{1}{24}R_{c,m-6} + \tfrac{1}{12}\sum_{j=-5}^{5} R_{c,m+j} + \tfrac{1}{24}R_{c,m+6} \qquad \text{(Eq. 4)}$$

and the anomaly is what remains:

$$A_{c,m} = R_{c,m} - \hat{\tau}_{c,m} \qquad \text{(Eq. 5)}$$

The moving average is undefined for the first and last six months, so anomalies cover 65 of the 77 months.

**Seasonal index and amplitude.** For each calendar month $k$, the seasonal index $S_{c,k}$ is the mean of $A_{c,m}$ over all months $m$ that fall in month $k$, re-centred so that the twelve values sum to zero. The amplitude summarises the strength of the pattern:

$$\text{amp}_c = \max_k S_{c,k} - \min_k S_{c,k} \qquad \text{(Eq. 6)}$$

**Permutation test.** If tone had no seasonal pattern, the calendar-month labels would carry no information. Shuffling them within each year $B = 2{,}000$ times and recomputing the amplitude gives the distribution expected by chance, and the p-value (Good, 2005) is

$$p_c = \frac{1 + \#\{b : \text{amp}_{c,b} \ge \text{amp}_{c}\}}{1 + B} \qquad \text{(Eq. 7)}$$

Because many countries are tested at once, p-values are converted to Benjamini-Hochberg q-values, and a country is called seasonal when $q < 0.05$ (Benjamini & Hochberg, 1995).

**Decomposition of the tone gap.** With $s_{ck}$ the share of country $c$'s events in CAMEO root category $k$ and $t_{ck}$ their mean tone (and $s_{gk}$, $t_{gk}$ the same for the world), the gap between a country and the world splits exactly into two parts, in the spirit of Kitagawa (1955):

$$\bar{T}_c - \bar{T}_g = \underbrace{\sum_k (s_{ck}-s_{gk})(t_{gk}-\bar{T}_g)}_{\text{mix: what happens there}} + \underbrace{\sum_k s_{ck}\,(t_{ck}-t_{gk})}_{\text{within: how it is reported}} \qquad \text{(Eq. 8)}$$

The *mix* part is negative when a country has more events in categories that are reported negatively everywhere (such as fighting). The *within* part is negative when a country's events are reported more negatively than events of the same category elsewhere. Expanding both sums gives $\sum_k s_{ck}t_{ck} - \bar{T}_g$, which is exactly $\bar{T}_c - \bar{T}_g$; Step 27 checks this numerically.

**Period change.** For the Philippines, the change in relative tone between two periods is compared with the change among its neighbours:

$$\Delta = \left(\bar{R}^{\,after}_{PH} - \bar{R}^{\,before}_{PH}\right) - \left(\bar{R}^{\,after}_{SEA} - \bar{R}^{\,before}_{SEA}\right) \qquad \text{(Eq. 9)}$$

$\Delta$ is descriptive: it shows whether coverage of the Philippines changed more than coverage of its neighbours, not what caused the change.

### 3.3 Design Decisions and Alternatives

Table 3 lists every design decision that could change the results, the reason for it, and the alternative that was considered. Where the alternative could be computed cheaply, it was, and the step that reports it is given.

**Table 3. Design decisions, the reason for each, the alternative considered, and where the alternative is tested.**

| # | Decision | Reason | Alternative considered | Tested in |
|---|---|---|---|---|
| 1 | Read each zip inside a Spark task (an RDD of file paths) | Spark's CSV reader cannot read zip archives, and unzipping would write about four times the raw data to disk | Unzip to disk, then `spark.read.csv` | Rejected; reason stated |
| 2 | Cache 19 fields as Parquet, partitioned by year | The raw files are read once; later steps read only the columns and years they need | Re-read the zips in every step | Runtime in Step 6 |
| 3 | Country = place of the event (`ActionGeo`) | The question is about how places are portrayed | Country of the first actor | Section 2.3 |
| 4 | Month = date added (`DATEADDED`) | The proposal measures tone by month of publication | Date of the event (`SQLDATE`) | Step 13 |
| 5 | Unit = event; pooled mean (Eq. 1) | Each event is one portrayal of a place | Article-weighted mean (Eq. 2), lead-paragraph events only, median | Step 21 |
| 6 | Confidence intervals by month-block bootstrap | Events in the same month are not independent | Event-level standard error | Step 19 |
| 7 | Minimum number of events per country | Countries with few events have unstable means | No threshold; three other thresholds | Step 17 |
| 8 | Seasonality on tone relative to the world (Eq. 3) | Removes global changes in volume and source mix | Raw monthly tone | Step 23 (Figure 13) |
| 9 | 2x12 moving-average detrending (Eq. 4) | Standard classical decomposition; keeps 65 of 77 months | Subtracting each year's mean | Step 24 |
| 10 | Permutation test with Benjamini-Hochberg control | No normality assumption; many countries tested at once | One-way ANOVA without correction | Step 24 |
| 11 | Drop events without a location | A country cannot be inferred from text that is not in the data | Assign the first actor's country | Step 12 |

### 3.4 Big Data Concepts Applied

The processing is built on Apache Spark's distributed data model and its SQL engine (Zaharia et al., 2016; Armbrust et al., 2015), following the course notebooks (Alis, n.d.). Table 4 lists the concepts applied and the techniques used beyond the course.

**Table 4. Course concepts and techniques beyond the course used in this study, and where each is applied.**

| Concept | How it is used here | Step |
|---|---|---|
| *Within the course* | | |
| Spark local mode and parallelism | One machine with 24 worker threads processes 2,335 files in parallel | 2, 6 |
| RDDs and `flatMap` | An RDD of file paths becomes an RDD of parsed rows, one task per file | 6 |
| Lazy evaluation and actions | All transformations are planned first; one Parquet write triggers the read | 6 |
| Columnar storage (Parquet) | Column pruning and partition pruning shown in the physical plan | 6 |
| Wide transformations and shuffles | Two `groupBy` aggregations build all analysis tables | 18, 26 |
| Broadcast join | The small country lookup table is broadcast instead of shuffling the event table | 16 |
| Approximate quantiles | `percentile_approx` for medians and percentiles over hundreds of millions of rows | 9, 18 |
| Persisting intermediate results | The filtered table is written to Parquet rather than cached in memory | 16 |
| *Beyond the course* | | |
| Bit-mask null profiling | Empty values in all 58 columns counted in one pass | 6, 11 |
| Month-block bootstrap | Confidence intervals that respect dependence within months | 19 |
| Classical seasonal decomposition | 2x12 moving average and seasonal index | 23, 24 |
| Permutation test and false discovery rate | Significance of seasonality across many countries | 24 |
| Kitagawa-style decomposition | Separating event mix from reporting tone | 27, 30 |

### 3.5 Supplementary Files and Reproducibility

This notebook generates the full report when run from top to bottom. Code that would make the report hard to read is kept in four supplementary Python files, which are imported in Step 1 and named at the start of every step that uses them (Table 5).

**Table 5. Supplementary files submitted with this notebook and the steps that use them.**

| File | Contents | Used in steps |
|---|---|---|
| `gdelt_io.py` | Column names of the 58-column format, the zip parser and the Spark reader | 4 to 7 |
| `gdelt_lookups.py` | CAMEO, QuadClass and location-type names, FIPS-to-ISO3 table, Southeast Asian list, dated Philippine events | 10, 16, 19 to 30 |
| `gdelt_analysis.py` | Summaries, null profile, bootstrap, moving average, seasonal index, permutation test, false discovery rate, decomposition, period change | 9 to 30 |
| `gdelt_plots.py` | Shared style and one function per figure | All figures |

All code follows PEP 8 (van Rossum et al., 2001), and every random step uses the seed 2026. The raw data are read in place and never copied. Parsed data are cached in a local `scratch` folder that is not submitted; with an empty `scratch` folder, Step 6 parses all raw files again, which took [[FILL: runtime in minutes]] minutes on Jojie.

### 3.6 Report Organisation

Section 4 (Steps 1 to 7) sets up Spark, reads the raw files and checks the parser. Section 5 (Steps 8 to 15) explores every raw field and investigates why values are missing. Section 6 (Steps 16 to 17) applies and counts the filters. Sections 7 to 10 (Steps 18 to 30) answer RQ1 to RQ4. Section 11 summarises the results and states the conclusion, Section 12 discusses limitations and future work, and the references close the report.

## 4. Environment and Data Ingestion

### Step 1. Imports, Paths and Parameters

Step 1 loads the libraries and the four supplementary files, and gathers every tunable parameter in one block, so that a reader can change any setting in one place. Table 6, after the code, states where each value is justified.

In [ ]:
# STEP 1: imports, paths, parameters

# 1. Standard library imports: os, time, zipfile, io, contextlib, datetime.
# 2. Third-party imports: numpy as np, pandas as pd, matplotlib.pyplot as plt, plotly.express as px, plotly.io as pio.
# 3. PySpark imports: SparkSession, pyspark.sql.functions as F, pyspark.sql.types as T, Window.
# 4. Import the four supplementary modules: gdelt_io, gdelt_lookups, gdelt_analysis, gdelt_plots.
#    (Optional: importlib.reload each one, so edits to the .py files are picked up without restarting.)

# 5. Paths (relative only, NO home directory, NO username):
#    RAW_DIR = the shared GDELT events folder (absolute path under /mnt/data/public is fine, it is not personal)
#    RAW_PATTERN = '*.export.CSV.zip'
#    SCRATCH_DIR = './scratch'; RAW_CACHE = './scratch/events_raw.parquet'; CLEAN_CACHE = './scratch/events_clean.parquet'
#    FIG_DIR = './figures'
#    Create SCRATCH_DIR and FIG_DIR if missing (os.makedirs with exist_ok).

# 6. Study window: START_DATE = '2013-04-01', END_DATE = '2019-08-26'.

# 7. Every tunable parameter in one commented block (one comment per line saying what it controls):
#    SEED = 2026                      # all random steps
#    N_BOOT = 2000                    # month-bootstrap resamples (Step 19)
#    N_PERM = 2000                    # permutation resamples (Step 24)
#    ALPHA = 0.05                     # significance / FDR level
#    MIN_EVENTS_TOTAL = 10_000        # ranking threshold; FINAL value set from Step 17 evidence
#    MIN_EVENTS_MONTH = 30            # every month must have at least this many events for seasonality
#    TOP_N = 15                       # countries shown at each end of the ranking
#    FOCUS_FIPS = 'RP'                # the Philippines (FIPS code, NOT 'PH')
#    SEA_FIPS = gdelt_lookups.SEA_FIPS
#    PH_SPLIT_MONTH = '2016-07'       # first full month of the new administration (Step 29)
#    SELECTED_FOR_CURVES = list of ~9 FIPS codes for Figure 15: fill in after Step 19
#                          (e.g. top 3 by volume, the most positive, the most negative, RP and 3 SEA neighbours)

# 8. Display settings: pd.set_option for max columns and float format (3 decimals).
# 9. Call gdelt_plots.set_style() so every figure shares one style.
# 10. Set pio.renderers.default = 'notebook' so plotly figures are embedded in the HTML export.
# 11. Print ONLY library versions (python, pyspark, pandas, numpy, matplotlib, plotly). Do not print paths that contain a home folder.

**Table 6. Tunable parameters, their values and where each value is justified.**

| Parameter | Value | Meaning | Justified in |
|---|---|---|---|
| `SEED` | 2026 | Seed of every random step | Reproducibility (Section 3.5) |
| `N_BOOT` | 2,000 | Month-bootstrap resamples | Step 19; percentile intervals are stable at this size (Efron & Tibshirani, 1993) |
| `N_PERM` | 2,000 | Permutations per country | Step 24; smallest attainable p-value is 1/2,001 |
| `ALPHA` | 0.05 | Significance level and false discovery rate | Step 24 |
| `MIN_EVENTS_TOTAL` | [[FILL: chosen value]] | Minimum events for a country to be ranked | Step 17 (Table 24, Figure 8) |
| `MIN_EVENTS_MONTH` | 30 | Minimum events in every month for a country's seasonality to be tested | Step 23 |
| `TOP_N` | 15 | Countries shown at each end of the ranking | Figure 9 |
| `FOCUS_FIPS` | `RP` | The Philippines (FIPS code) | Section 2.3 |
| `PH_SPLIT_MONTH` | 2016-07 | First full month after the presidential inauguration of 30 June 2016 (Dizon, 2016) | Step 29 |

### Step 2. Starting Spark

Spark runs in local mode on the Jojie server, where one process uses several CPU cores as parallel workers. The server has 32 cores and 251 GB of memory and is shared with other users, so the session uses 24 cores and 64 GB, leaving capacity for others. The number of shuffle partitions is set to 96, about four per core, instead of Spark's default of 200, which would create many small tasks for the modest aggregate tables built later.

In [ ]:
# STEP 2: create the SparkSession

# 1. BEFORE building the session, set the environment variable SPARK_LOCAL_IP to '127.0.0.1'.
#    Why: otherwise Spark prints a warning containing the machine's hostname, which contains the username (anonymity).
# 2. Build the session with SparkSession.builder:
#    - appName 'gdelt_tone'
#    - master 'local[24]'  (24 of 32 cores; comment why: shared server)
#    - spark.driver.memory '64g'  (local mode: the driver is the executor)
#    - spark.sql.shuffle.partitions 96  (about 4 x cores; default 200 makes many tiny tasks)
#    - spark.local.dir './scratch/spark_tmp'  (spill files stay in the project, not in /tmp)
#    - spark.sql.session.timeZone 'UTC'
#    - spark.sql.execution.arrow.pyspark.enabled 'true'  (faster toPandas)
# 3. spark.sparkContext.setLogLevel('ERROR') so warnings do not clutter the report.
# 4. Keep a handle: sc = spark.sparkContext.
# 5. Print ONLY: spark.version and sc.defaultParallelism.
# 6. NEVER put `spark` or `sc` alone on the last line of a cell (their display shows the hostname/username).

### Step 3. Smoke Tests

Step 3 checks, in a few seconds, the two things the rest of the notebook depends on: that a raw zip file opens and has 58 fields, and that the world map can be drawn and saved.

In [ ]:
# STEP 3: smoke tests

# 1. Open the smallest raw zip with zipfile.ZipFile; assert it contains exactly ONE member; read its first line; assert it splits into 58 tab-separated fields. Print "zip OK, 58 fields".
# 2. Draw a tiny plotly choropleth with 3 countries (locations as ISO-3 codes, locationmode 'ISO-3'). Show it.
# 3. Try fig.write_image('figures/_smoke.png') inside try/except.
#    - If it works: STATIC_MAP_OK = True.
#    - If it fails (kaleido 1.x needs a Chrome binary): STATIC_MAP_OK = False; the map is then saved only as HTML (fig.write_html) and shown inline.
#    Print which case happened (do not print the exception text, it may contain paths).
# 4. Delete the smoke-test image file afterwards.

### Step 4. Inventory of the Raw Files

Step 4 lists the files the study reads, confirms that they cover every day of the study period except the four known gaps, and shows one raw record with all 58 fields so that the raw format is visible (Tables 7 and 8).

**Table 7. Raw daily files per year: number of files, days without a file, and compressed size.**

**Table 8. One raw GDELT 1.0 record (the first line of the 1 April 2013 file), with all 58 field names and values.**

In [ ]:
# STEP 4: raw file inventory  (uses list_raw_files from gdelt_io.py)

# 1. raw_paths = list_raw_files(RAW_DIR): sorted list of files matching RAW_PATTERN.
# 2. Assert len(raw_paths) == 2335. Print the count, the first and the last file NAME (basename only).
# 3. Parse the date from each file name (first 8 characters, format %Y%m%d).
# 4. Build the full expected daily range START_DATE..END_DATE with pd.date_range; list dates with no file.
#    Assert the missing list equals the 4 known dates; print them.
# 5. Compressed sizes: os.path.getsize per file. Print total in GiB (should be close to the 24G from du).
# 6. Table 7: group by year -> number of files, days missing that year, total compressed GiB, mean MB per file. Add a total row. Display.
# 7. Also list the OTHER files in RAW_DIR that do NOT match the pattern (yearly zips, MASTERREDUCED, index.html, filesizes, md5sums): count by type, print. These are excluded (Section 2.1 says why).
# 8. Table 8: read the FIRST line of the 20130401 zip with zipfile (no Spark needed), split on tab, show a two-column DataFrame (field name from gdelt_io.COLUMN_NAMES, value). Display all 58 rows.

The study reads [[FILL: 2,335]] daily files totalling [[FILL: GiB]] GiB compressed (Table 7). Only four days lack a file, all in early 2014; because tone is averaged over events, these gaps do not bias monthly means, and monthly volume is expressed per available day (Step 8). Table 8 shows the raw format: many actor fields are empty, the event date (`SQLDATE` 20030404) lies ten years before the date added (20130401), and Australia appears under the FIPS code `AS`. Each of these observations becomes a design decision in Section 2.3.

### Step 5. Parser Design and Validation

Spark's built-in CSV reader cannot read zip archives, and unzipping the files would copy the raw data. The function `build_events_df` in `gdelt_io.py` therefore builds an RDD of **file paths**, one task per file, and the function `parse_zip` opens each archive inside its task and yields one row per line (`flatMap`). Only the 19 fields of Table 1 are kept, together with three audit fields: `n_fields`, the number of tab-separated fields (which must be 58); `empty_mask`, a 64-bit integer whose bit *i* is 1 when field *i* is empty, so that empty values in all 58 columns can be counted without keeping them (Step 11); and `file_date`, the date in the file name. Event codes are kept as text, because codes such as `04` and `043` have leading zeros.

The parser is first tested on three files from 2013, 2016 and 2019, whose line counts were taken independently with `wc -l` on Jojie.

**Table 9. Parser validation: lines counted by `wc -l` on Jojie, by plain Python, and rows parsed by Spark, for three files from 2013, 2016 and 2019.**

**Table 10. First five rows of the parsed events table.**

In [ ]:
# STEP 5: build and test the parser  (uses parse_zip, build_events_df from gdelt_io.py)

# What parse_zip(path) in gdelt_io.py must do (write it there, with a docstring):
#   a. file_date = int of the first 8 characters of the file name
#   b. open the zip, open its single member, iterate over lines (do not read the whole file into a list)
#   c. decode each line as UTF-8 with errors='replace', strip the trailing newline only
#   d. fields = line.split('\t'); n_fields = len(fields)
#   e. empty_mask = sum of (1 << i) for every i where fields[i] == ''  (only when n_fields == 58; else -1)
#   f. yield a tuple: the 19 kept fields as strings (None if the row is too short), n_fields, empty_mask, file_date
# What build_events_df(spark, paths) must do:
#   g. rdd = sc.parallelize(paths, len(paths)).flatMap(parse_zip)
#   h. spark.createDataFrame(rdd, schema) with an explicit StructType (strings + 3 integer/long audit fields)
#   i. cast in Spark (not Python): event_id long; sql_date int; is_root int; event_code / event_base_code / event_root_code string (keep leading zeros);
#      quad_class int; goldstein double; num_mentions / num_sources / num_articles int; avg_tone double;
#      geo_type int; country_fips string with '' turned into null (F.when(F.trim(col) == '', None)); adm1 string; geo_fullname string; lat / lon double; date_added int
#   j. derived columns: date_added_dt = to_date of date_added as 'yyyyMMdd'; sql_date_dt the same for sql_date;
#      year_added, month_added (1-12), ym_added ('yyyy-MM' string); lag_days = datediff(date_added_dt, sql_date_dt)
#   k. return the DataFrame (lazy: nothing is read yet)

# In the notebook cell:
# 1. TEST_FILES = the three files 20130401, 20160615, 20190826 (paths built from RAW_DIR).
# 2. WC_COUNTS = {20130401: 27758, 20160615: 240023, 20190826: 157071}  # from `wc -l` on jojie, 15 Nov 2026
# 3. Reference count with plain Python: count lines per file with zipfile (slow but obviously correct).
# 4. test_df = build_events_df(spark, TEST_FILES); count rows per file_date with groupBy.
# 5. Table 9: file, wc -l, Python count, Spark count, match (True/False). Display. Assert all True.
# 6. Assert every test row has n_fields == 58 and date_added == file_date (count the exceptions, print 0).
# 7. Table 10: test_df.limit(5).toPandas(), with the most important columns. Display.

Spark parsed exactly the same number of rows as `wc -l` and plain Python counted in all three files (Table 9), every row had 58 fields, and every `DATEADDED` matched its file's date. The parser is therefore applied to all files in Step 6.

### Step 6. One Spark Pass over All 2,335 Files

This is the only step that reads the raw files. The transformations that define the table are lazy: Spark only plans them. The single action, writing the table to Parquet, triggers the read, with one task per file running on 24 cores. The output is partitioned by year of publication, so later steps read only the columns and years they need. If the Parquet cache already exists, the step reads it instead of parsing again.

In [ ]:
# STEP 6: full ingestion  (uses list_raw_files, build_events_df, write_parquet from gdelt_io.py)

# 0. BEFORE the first full run, check free space in the home folder in a terminal (df -h ~ and, if the course uses quotas, quota -s).
#    The two Parquet caches may need tens of GB. If space is short, write only RAW_CACHE and in Step 16 keep events_df as a filtered view
#    of it instead of writing CLEAN_CACHE (comment why).
# 1. If RAW_CACHE exists AND contains a _SUCCESS file: print "reading cache" and skip to item 5.
# 2. Else: t0 = time.perf_counter(); events_build_df = build_events_df(spark, raw_paths).
# 3. write_parquet(events_build_df, RAW_CACHE, partition_col='year_added') with mode 'overwrite'.
#    (Comment: the Parquet write is the ONE action that triggers reading all 2,335 zips; ~one task per file.)
# 4. Print elapsed minutes (rounded to 1 decimal).
# 5. events_raw_df = spark.read.parquet(RAW_CACHE).
# 6. Print events_raw_df.rdd.getNumPartitions() and show events_raw_df.printSchema().
# 7. Evidence of pruning (Integration of Learnings): capture the physical plan of
#    events_raw_df.filter(year_added == 2016).select('avg_tone') as TEXT with contextlib.redirect_stdout + io.StringIO,
#    REPLACE os.path.expanduser('~') and os.getcwd() in that text with '<project>' (anonymity), then print only the
#    lines containing 'PartitionFilters' and 'ReadSchema'.

The full parse took [[FILL]] minutes and produced a Parquet table of [[FILL]] partitions. The physical plan printed above shows the benefit of the columnar format: a query for the 2016 tone values reads only the `avg_tone` column (`ReadSchema`) and only the 2016 folder (`PartitionFilters`), which is why every later step runs in seconds or minutes rather than re-reading 24 GB.

### Step 7. Acceptance Checks on the Full Table

Before any analysis, the full parsed table is checked against what the raw files guarantee (Table 11).

**Table 11. Acceptance checks on all parsed rows: each check, the expected result, the observed result, and whether it passed.**

In [ ]:
# STEP 7: acceptance checks (ONE aggregation over events_raw_df, F.count / F.sum(F.when(...)))

# Compute in a single .agg(...):
# 1. N_RAW = total rows (store as a Python int; it is the 100% base of the filtering table in Step 16).
# 2. number of distinct file_date values  -> expected 2335
# 3. rows with n_fields != 58             -> expected 0 (if not 0, they are removed in Step 16 and counted there)
# 4. rows with date_added != file_date    -> expected 0 (DATEADDED must equal the file's day)
# 5. rows with avg_tone null after cast   -> report
# 6. rows with avg_tone outside [-100, 100] -> expected 0
# 7. min and max of date_added_dt          -> expected 2013-04-01 and 2019-08-26
# 8. Also: rows per file -> min, median, max (groupBy file_date then describe), to spot empty or truncated files.
# Build Table 11 as a pandas DataFrame (check, expected, observed, passed) and display it. Print N_RAW with thousands separators.

The parsed table holds [[FILL: N_RAW]] events from [[FILL: 2,335]] files. [[FILL: State that all checks passed, or name the check that did not and how Step 16 handles it.]]

## 5. Exploratory Data Analysis and Null Investigation

Steps 8 to 15 describe every part of the raw data: volume over time (Step 8), the numeric fields (Step 9), the categorical fields (Step 10), empty values in all 58 columns (Step 11) and why events lack a location (Step 12), the lag between an event and its publication (Step 13), and duplicate identifiers (Step 14). Each finding leads to a design decision, summarised in Table 21 (Step 15).

### Step 8. Volume over Time

**Table 12. Events per year, events per available day, and share of all parsed events.**

In [ ]:
# STEP 8: daily and yearly volume

# 1. daily_pd = events_raw_df.groupBy('date_added_dt').count() -> toPandas, sorted by date.
# 2. Reindex daily_pd onto the full daily range so the 4 missing days appear as NaN (not 0).
# 3. Table 12: per year -> events, days with a file, events per available day, % of all events. Add a total row. Display.
# 4. FIGURE 2 (plot_daily_volume in gdelt_plots.py):
#    - thin grey line = daily events; thick line = 30-day centred rolling mean
#    - small red markers along the x-axis at the 4 missing days
#    - y-axis in thousands of events, labelled; x-axis = publication date
#    - save figures/fig02_daily_volume.png

**Figure 2. Number of GDELT events added per day, 1 April 2013 to 26 August 2019. Grey: daily count; dark line: 30-day centred mean; red ticks: the four days with no file.**

Figure 2 and Table 12 show that daily volume ranged from about [[FILL]] to [[FILL]] thousand events and was not stable over the period: [[FILL: describe the level changes visible in Figure 2, e.g. a rise or a step in a given year]]. Such changes reflect GDELT's own source list as much as world events (Wang et al., 2016). Because a change in sources can also shift average tone, seasonality is measured on each country's tone **relative to the global mean of the same month** (Eq. 3), which cancels changes that affect all countries at once.

### Step 9. Numeric Fields

**Table 13. Summary statistics of the five numeric fields over all parsed events (percentiles computed with `percentile_approx`).**

In [ ]:
# STEP 9: numeric summary  (uses numeric_summary from gdelt_analysis.py)

# 1. NUMERIC_COLS = ['avg_tone', 'goldstein', 'num_mentions', 'num_sources', 'num_articles'].
# 2. numeric_summary(events_raw_df, NUMERIC_COLS): ONE .agg with count (non-null), mean, stddev, min, max,
#    and F.percentile_approx(col, [0.01, 0.25, 0.5, 0.75, 0.99], accuracy=10000) for each column. Return a tidy pandas table.
# 3. Display Table 13 (one row per field).
# 4. Histogram of avg_tone computed IN SPARK: bin = F.floor(avg_tone / 0.5) * 0.5, groupBy bin, count, toPandas.
# 5. FIGURE 3 (plot_tone_hist): bar histogram for -20..+20, vertical lines for mean and median (labelled in legend),
#    a text note with the % of events outside -20..+20. Save figures/fig03_tone_hist.png.

**Figure 3. Distribution of AvgTone over all parsed events, in bins of 0.5 tone points. Dashed line: mean; dotted line: median. [[FILL]]% of events fall outside the plotted range of -20 to +20.**

Global news tone is [[FILL: mostly negative / centred near zero]]: the mean is [[FILL]] and the median [[FILL]] (Table 13), and [[FILL]]% of events have a negative tone (Figure 3). Because the baseline is not zero, every country is compared with the global mean rather than with neutral tone. `NumArticles` is highly skewed (median [[FILL]], 99th percentile [[FILL]]), so the article-weighted mean (Eq. 2) can differ from the pooled mean; Step 21 tests whether this matters.

### Step 10. Categorical Fields

**Table 14. Events and mean tone by QuadClass.**

**Table 15. Events, share and mean tone by CAMEO root category (01 to 20).**

**Table 16. Events and share by location precision (`ActionGeo_Type`).**

In [ ]:
# STEP 10: categorical distributions  (uses QUAD_CLASS_NAMES, CAMEO_ROOT_NAMES, GEO_TYPE_NAMES from gdelt_lookups.py)

# 1. Table 14: groupBy quad_class -> count, mean avg_tone; add names and % share. Display.
# 2. Table 15: groupBy event_root_code -> count, mean avg_tone; add root names and % share; sort by code. Display.
#    Keep this as global_root_pd (reused in Step 27).
# 3. Table 16: groupBy geo_type -> count; add names and % share. Display.
# 4. Also print: share of root events (is_root == 1); number of distinct country_fips values (non-null).
# 5. FIGURE 4 (plot_tone_by_root): horizontal bars of mean tone per CAMEO root (20 bars, labelled "04 Consult" etc.),
#    sorted by tone, coloured by QuadClass (4 colours, legend), vertical line at the global mean.
#    Save figures/fig04_tone_by_root.png.

**Figure 4. Mean AvgTone of all events in each CAMEO root category, coloured by QuadClass. The vertical line marks the global mean.**

Verbal cooperation is the most common type of event ([[FILL]]% of events, Table 14), while material conflict accounts for [[FILL]]%. Figure 4 shows that mean tone differs sharply between categories, from [[FILL: most negative category and value]] to [[FILL: most positive category and value]]. A country's mean tone therefore depends partly on *which kinds* of events happen there, which RQ3 separates from *how* they are reported. Table 16 shows that [[FILL]]% of events are located at the country level only, [[FILL]]% at a city or region, and [[FILL]]% have no location at all; the last group is investigated in Step 12.

### Step 11. Empty Values in All 58 Raw Columns

GDELT writes a missing value as an empty field rather than as a null marker. Using the bit mask built during parsing, Step 11 counts empty values in all 58 columns in a single pass.

**Table 17. Number and share of empty values in each of the 58 raw columns (all parsed rows), and whether the column is used in this study.**

In [ ]:
# STEP 11: null profile of every raw column  (uses null_profile from gdelt_analysis.py)

# 1. null_profile(events_raw_df): ONE .agg with 58 expressions,
#    F.sum(F.shiftright('empty_mask', i).bitwiseAND(1)) for i in 0..57, over rows with n_fields == 58.
# 2. Return a pandas table: index, column name (COLUMN_NAMES), empties, % empty, used in study (True if in KEEP_COLUMNS).
# 3. Display Table 17 sorted by % empty, descending (all 58 rows).
# 4. Separately, for the USED columns: count nulls created by the CAST (non-empty text that failed to parse as a number).
#    Print them; expected 0.

Empty values are concentrated in the actor columns (Table 17): [[FILL: e.g. the second actor's fields are empty in X% of rows]], because many events have only one identifiable actor, and actor attributes such as religion or ethnicity apply to few actors. Among the 19 columns this study uses, only the location fields have a notable share of empty values ([[FILL]]% for `ActionGeo_CountryCode`); every other used field is complete, and no non-empty value failed to convert to its numeric type.

### Step 12. Why Some Events Have No Location

Before deciding what to do with events that have no country, Step 12 investigates why the country is missing.

**Table 18. Events by location precision (`ActionGeo_Type`) and whether the country code is empty.**

In [ ]:
# STEP 12: investigate the empty country code BEFORE deciding to drop it

# 1. Table 18: crosstab geo_type x (country_fips is null) with counts and row %. Display.
#    Expected pattern to CHECK (do not assume): country empty <=> geo_type == 0 ("no location").
# 2. Among geo_type > 0: count rows with null lat or lon (expected ~0).
# 3. Share of unlocated events per publication month (groupBy ym_added): keep for Figure 5a.
# 4. Compare located vs unlocated events: mean avg_tone overall and per QuadClass, and mean num_articles. Keep for Figure 5b and print the table.
# 5. FIGURE 5 (plot_unlocated, two panels):
#    (a) line: % of events with no location per month;
#    (b) grouped bars: mean tone of located vs unlocated events for each QuadClass.
#    Save figures/fig05_unlocated.png.
# 6. Print the decision variables: number and % of events that will be dropped in Step 16 for this reason.

**Figure 5. Events without a location: (a) share of events per publication month with no country code; (b) mean AvgTone of located and unlocated events, by QuadClass.**

Table 18 shows that the country code is empty [[FILL: exactly / almost exactly]] when `ActionGeo_Type` is 0, GDELT's code for "no location". The value is therefore not lost; it never existed. The geocoder assigns a place only when the article names one it can resolve, and many events, especially statements and other verbal events, are reported without a place: [[FILL]]% of unlocated events are verbal, against [[FILL]]% of located events. Figure 5a shows that the share of unlocated events [[FILL: was stable at about X% / changed from X% to Y%]] over the period. Figure 5b shows that unlocated events are reported with [[FILL: similar / more negative / more positive]] tone than located events of the same QuadClass (difference of [[FILL]] points).

These events are **removed** in Step 16. A country cannot be imputed, because the text that might name it is not part of the dataset. Assigning the first actor's country instead would answer a different question (who acts rather than where), and imputing tone is irrelevant, since the tone is present and only the location is missing. [[FILL: If Figure 5b shows a clear difference, add: "Because unlocated events are reported slightly more negatively, removing them may raise all countries' tone by a similar amount; the ranking between countries is unaffected, and the issue is listed in Table 35."]]

### Step 13. Event Date versus Publication Date

**Table 19. Lag between the date of the event (`SQLDATE`) and the date it was added (`DATEADDED`): events and mean tone in each lag bucket.**

In [ ]:
# STEP 13: lag between event and publication

# 1. Buckets of lag_days: negative (data error), 0, 1, 2-7, 8-30, 31-365, more than 365, null (unparseable SQLDATE).
# 2. Table 19: per bucket -> events, %, mean avg_tone. Display.
# 3. FIGURE 6 (plot_lag): horizontal bars of % of events per bucket, with the mean tone printed at the end of each bar.
#    Save figures/fig06_lag.png.

**Figure 6. Share of events by the delay between the date the event took place and the date the article was processed.**

[[FILL]]% of events were added on the day they took place or the next day (Table 19), but [[FILL]]% refer to events more than a month old and [[FILL]]% to events more than a year old, as with the 2003 event in the first file of the study. Grouping by `SQLDATE` would place this coverage in the wrong month, so all time analysis uses `DATEADDED`. Old events are kept, because they are part of how a country was covered in the month the article appeared. [[FILL: Number of events with a negative lag, and whether they were kept or removed.]]

### Step 14. Duplicate Event Identifiers

**Table 20. Duplicated `GLOBALEVENTID` values: identifiers that appear more than once and the rows involved.**

In [ ]:
# STEP 14: duplicates (a shuffle on event_id only: cheap)

# 1. dup_ids_df = events_raw_df.groupBy('event_id').count().filter(count > 1).
# 2. Table 20: number of duplicated IDs, total rows involved, extra rows (rows minus IDs), % of N_RAW. Display.
# 3. If extra rows > 0: show 3 example IDs with all their rows (are they identical, or same ID different day?).
#    Decide the rule: keep the row with the earliest date_added (state it). Store N_DUP_EXTRA for Step 16.
# 4. If extra rows == 0: print "no duplicate IDs; no de-duplication needed".

[[FILL: Either "No identifier appears more than once, so no de-duplication is needed." or "X identifiers appear more than once, adding Y extra rows (Z% of all rows); they are [identical copies / the same event added on different days], and only the row with the earliest DATEADDED is kept in Step 16."]]

### Step 15. Summary of the Exploration

**Table 21. Findings of the exploratory analysis and the design decision each one led to.**

| # | Finding | Evidence | Decision | Applied in |
|---|---|---|---|---|
| 1 | Four days have no file | Table 7 | Monthly volume expressed per available day; tone unaffected | Step 22 |
| 2 | Daily volume changes over the period | Figure 2, Table 12 | Seasonality measured on tone relative to the world (Eq. 3) | Step 23 |
| 3 | Mean global tone is [[FILL]], not zero | Table 13, Figure 3 | Countries compared with the global mean | Steps 19 to 20 |
| 4 | `NumArticles` is highly skewed | Table 13 | Article-weighted mean tested as an alternative | Step 21 |
| 5 | Tone differs strongly by CAMEO category | Figure 4, Table 15 | Tone gap decomposed into mix and within parts | Step 27 |
| 6 | Country is empty exactly when location type is 0 | Table 18, Figure 5 | Unlocated events removed after investigation | Step 16 |
| 7 | [[FILL]]% of events are more than a month old when added | Table 19, Figure 6 | Time measured by `DATEADDED` | Step 18 |
| 8 | [[FILL: duplicates finding]] | Table 20 | [[FILL: decision]] | Step 16 |
| 9 | Actor fields often empty | Table 17 | Location taken from the event, not the actor | Section 2.3 |

## 6. Data Filtering

### Step 16. Filtering, with Counts at Every Step

The filters are applied in the order of Table 23, and the number of rows is counted after each one. Before the country filter, all FIPS codes that have no entry in the lookup table are listed with their number of events (Table 22), so that gaps in the lookup table are not mistaken for bad data. The lookup table is joined with a broadcast join: it has about 250 rows, so Spark copies it to every task instead of shuffling the event table.

**Table 22. FIPS country codes present in the data with no entry in the lookup table, and their number of events.**

**Table 23. Filtering steps: the rule, its justification, rows remaining, rows removed, and the share kept relative to the previous step and to all parsed rows.**

In [ ]:
# STEP 16: apply the filters in order and COUNT after each one  (broadcast join with the FIPS lookup)

# Start from events_raw_df. After EACH filter store the row count (one .count() per step is fine on Parquet).
#   F0  all parsed rows (= N_RAW from Step 7)
#   F1  n_fields == 58                                  # malformed lines
#   F2  avg_tone not null and between -100 and 100      # outcome must be valid
#   F3  de-duplicated with the rule from Step 14        # skip (same count) if there were no duplicates
#   F4  country_fips not null                           # located events only (justified in Step 12)
#   F5  country_fips present in FIPS_TO_ISO3            # real countries/territories only
#       - build lookup_df = spark.createDataFrame from FIPS_TO_ISO3 (fips, iso3, country_name)
#       - inner join with F.broadcast(lookup_df) on country_fips == fips   (comment: small table -> broadcast, no shuffle of the big table)
#       - BEFORE the join, list codes NOT in the lookup with their event counts -> Table 22. Display.
#         If a code with many events is unmapped, add it to the lookup (it is a lookup gap, not bad data) and re-run.
#   F6  date_added_dt between START_DATE and END_DATE   # expected to remove 0; keep as an explicit check

# 1. Table 23 columns: step, rule, justification (one short phrase), rows remaining, rows removed,
#    % kept of previous step, % kept of all parsed rows. Display with thousands separators.
# 2. FIGURE 7 (plot_funnel): horizontal bars of rows remaining per step (top = F0), labelled with the count and the % of F0.
#    Save figures/fig07_filter_funnel.png.
# 3. Keep only the analysis columns (event_id, date_added_dt, year_added, month_added, ym_added, country_fips, iso3, country_name,
#    event_root_code, quad_class, avg_tone, num_articles, is_root, geo_type, adm1, geo_fullname, lat, lon) and write events_df to CLEAN_CACHE
#    (Parquet, partitioned by year_added; skip if it exists), then events_df = spark.read.parquet(CLEAN_CACHE).
#    (Comment: writing beats .cache() here because the table is larger than comfortable driver memory, and it survives a kernel restart.)
# 4. Store N_CLEAN = final row count.

**Figure 7. Rows remaining after each filtering step, from all parsed GDELT events to the located, valid events used in the analysis.**

Of the [[FILL: N_RAW]] parsed events, [[FILL: N_CLEAN]] ([[FILL]]%) remain after filtering (Table 23, Figure 7). The largest reduction comes from removing events without a location ([[FILL]] rows, [[FILL]]% of the previous step), as anticipated in Step 12. Malformed rows removed [[FILL]] rows, invalid tone [[FILL]], duplicates [[FILL]], and codes that do not denote a country [[FILL]] ([[FILL]]% of located events; Table 22 lists them, [[FILL: e.g. ocean and region codes]]). The date-range check removed [[FILL: 0]] rows, confirming that every file contains only its own day's additions.

### Step 17. Minimum Number of Events per Country

A country with few events has an unstable mean tone, so only countries above a minimum number of events are ranked. Step 17 chooses that minimum from the data, by comparing how many countries and events four candidate thresholds keep against how precise the resulting means are.

**Table 24. Countries and events kept under four candidate thresholds, and the widest 95% interval among the countries kept.**

In [ ]:
# STEP 17: choose MIN_EVENTS_TOTAL

# 1. country_n_pd = events_df.groupBy('country_fips').agg(count, mean tone, stddev tone) -> toPandas.
# 2. Naive standard error per country = stddev / sqrt(n); 95% half-width = 1.96 * SE (the month-bootstrap CI in Step 19 is the one reported; this is only to choose the threshold).
# 3. Table 24: for thresholds 1,000 / 5,000 / 10,000 / 50,000 events ->
#    countries kept, % of countries, events kept, % of N_CLEAN, largest 95% half-width among kept countries. Display.
# 4. FIGURE 8 (plot_threshold): scatter of 95% half-width (y, log) vs number of events (x, log), one dot per country,
#    vertical dashed lines at the 4 thresholds, the chosen one highlighted. Save figures/fig08_threshold.png.
# 5. Set MIN_EVENTS_TOTAL to the chosen value (re-state it here even though Step 1 has a default; keep both equal).
# 6. eligible_fips = list of countries with n >= MIN_EVENTS_TOTAL. Print how many countries and what % of events.

**Figure 8. Width of the 95% confidence half-interval of mean tone against the number of events per country (log-log scale). Dashed lines mark the candidate thresholds; the solid line marks the chosen one.**

Figure 8 shows that the uncertainty of a country's mean tone falls steeply with its number of events. A threshold of [[FILL: chosen value]] events keeps every country's 95% half-interval below [[FILL]] tone points while retaining [[FILL]] of [[FILL]] countries ([[FILL]]%) and [[FILL]]% of all located events (Table 24). Countries below the threshold are shown in grey on the map but are not ranked.

## 7. RQ1: Which countries are covered most positively and most negatively?

### Step 18. The Country-by-Month Table

Every result in Sections 7, 8 and 10 is computed from one small table of sums per country and publication month, built with a single Spark `groupBy`. This is a wide transformation: rows with the same country and month must be brought together, so Spark shuffles the data once. The table stores sums (number of events, sum of tone, sum of tone weighted by articles) rather than means, so that any weighting, period or bootstrap resample can be recomputed exactly in pandas without touching the event table again.

**Table 25. First five rows of the country-by-month table of event counts and tone sums.**

In [ ]:
# STEP 18: country x month sums  (uses monthly sums; results saved to scratch only)

# 1. country_month_df = events_df.groupBy('country_fips', 'iso3', 'country_name', 'ym_added').agg(
#        n_events = count,
#        sum_tone = sum(avg_tone),
#        sum_tone_sq = sum(avg_tone^2),
#        sum_articles = sum(num_articles),
#        sum_tone_art = sum(avg_tone * num_articles),
#        n_root = sum(is_root),
#        sum_tone_root = sum(avg_tone * is_root))
# 2. country_month_pd = toPandas(); print its shape (expected about 250 countries x 77 months, minus empty cells).
# 3. global_month_pd = the same sums grouped by ym_added only (all located events). Add days_available per month (from Step 4) and events_per_day.
# 4. Save both to scratch/ as CSV (NOT submitted; used to avoid recomputation if you re-run later cells).
# 5. Display Table 25 = country_month_pd.head().
# 6. Also compute, in Spark, median tone per country: groupBy country_fips -> percentile_approx(avg_tone, 0.5, 10000). Keep as median_pd for Step 21.

The aggregate has [[FILL]] rows ([[FILL]] countries by up to 77 months), small enough to analyse in memory, compared with [[FILL: N_CLEAN]] rows in the event table.

### Step 19. Ranking with Confidence Intervals

Each country's pooled mean tone (Eq. 1) is computed from the monthly sums, with a 95% month-block bootstrap interval, and the eligible countries are ranked from most positive to most negative.

**Table 26. The 15 most positively and the 15 most negatively covered countries: number of events, pooled mean tone (Eq. 1) with its 95% month-bootstrap interval, and difference from the global mean.**

In [ ]:
# STEP 19: country summary and ranking  (uses pooled_means and bootstrap_month_ci from gdelt_analysis.py)

# 1. country_summary_pd = pooled_means(country_month_pd): per country -> N events, Eq. 1 pooled mean,
#    Eq. 2 article-weighted mean, root-only mean. Merge median_pd.
# 2. global_mean = sum of sum_tone / sum of n_events over all located events. Print it.
# 3. bootstrap_month_ci(country_month_pd, n_boot=N_BOOT, seed=SEED): for each country, resample its months
#    with replacement (same resampled month set for all countries in one draw: draw month indices once per b),
#    recompute pooled mean = sum(sum_tone)/sum(n_events); return 2.5% and 97.5% percentiles.
#    Vectorise with numpy (countries x months arrays), do not loop over events.
# 4. Also the naive event-level 95% half-width (1.96*sd/sqrt(n)) for comparison: print the median ratio
#    bootstrap width / naive width (shows why the naive CI is too narrow).
# 5. Keep eligible countries only (Step 17). Add rank (1 = most positive) and diff_from_global.
# 6. Table 26: top TOP_N and bottom TOP_N, columns: rank, country, ISO3, events, mean tone, CI low, CI high, diff from global. Display.
# 7. FIGURE 9 (plot_ranking): dot plot, two stacked panels (most positive 15 / most negative 15) OR one panel with a gap,
#    dots = mean, horizontal whiskers = 95% CI, vertical dashed line = global mean, country names on the y-axis,
#    diverging colour by diff from global. Save figures/fig09_ranking.png.
# 8. Now fill SELECTED_FOR_CURVES in Step 1 using these results (do not hard-code a guess before seeing them).

**Figure 9. The 15 countries with the most positive and the 15 with the most negative pooled mean news tone, April 2013 to August 2019. Dots: mean AvgTone of all located events; whiskers: 95% month-bootstrap interval; dashed line: global mean. Only countries with at least [[FILL: threshold]] events are ranked.**

Across all located events, the global mean tone is [[FILL]]. [[FILL: Country]] is the most positively covered country (mean [[FILL]], 95% interval [[FILL]] to [[FILL]]), followed by [[FILL]] and [[FILL]]; [[FILL: country]] is the most negatively covered ([[FILL]], [[FILL]] to [[FILL]]), followed by [[FILL]] and [[FILL]] (Table 26, Figure 9). The intervals of the top and bottom groups [[FILL: do not overlap]], so the difference between the two ends of the ranking is far larger than its uncertainty. [[FILL: Describe one pattern visible in Figure 9, e.g. which regions or kinds of country dominate each end, without explaining causes; RQ3 addresses causes.]] The month-bootstrap intervals are a median of [[FILL]] times wider than naive event-level intervals, which shows that treating events as independent would have overstated the precision of the ranking.

### Step 20. World Map of News Tone

Figure 10 places the same measure on a map, so that regional patterns are visible at a glance. Colours are centred on the global mean, so red means more negative than the world and blue more positive.

In [ ]:
# STEP 20: choropleth  (plot_world_map in gdelt_plots.py)

# 1. map_pd = country_summary_pd with iso3, country_name, events, mean tone, CI, eligible flag.
# 2. plotly.express.choropleth: locations = iso3, locationmode 'ISO-3', color = mean tone,
#    diverging colour scale (e.g. 'RdBu'), color_continuous_midpoint = global_mean, symmetric range (clip at the 2nd/98th percentile of eligible countries).
# 3. Countries below the threshold: grey (second trace, or set color NaN and note it in the caption).
# 4. hover: country name, events (thousands separator), mean tone (2 decimals), 95% CI.
# 5. Natural Earth projection, no frame, colour bar titled "Mean AvgTone".
# 6. fig.show(); fig.write_html('figures/fig10_world_map.html'); if STATIC_MAP_OK also write_image PNG.

**Figure 10. Mean news tone of events located in each country, April 2013 to August 2019. Red: more negative than the global mean; blue: more positive; grey: fewer than [[FILL: threshold]] events. Hover over a country for its value, number of events and 95% interval.**

[[FILL: Two sentences describing the geographic pattern in Figure 10, e.g. which regions appear mostly red or blue and any country that stands out from its neighbours.]]

### Step 21. Robustness of the Ranking

A ranking is only useful if it does not depend on arbitrary choices. Step 21 recomputes it under four alternatives: weighting events by number of articles (Eq. 2), keeping only lead-paragraph events, using the median instead of the mean, and ranking each full calendar year (2014 to 2018) separately.

**Table 27. Spearman rank correlation between the main ranking and rankings under alternative choices, and how many of the top and bottom ten countries stay in the top and bottom ten.**

In [ ]:
# STEP 21: robustness of RQ1

# 1. Alternative rankings of the eligible countries: (a) article-weighted mean (Eq. 2), (b) root events only, (c) median, (d) each calendar year 2014 to 2018 separately (full years only; computed from country_month_pd).
# 2. For each alternative: Spearman correlation with the main ranking (pandas .corr(method='spearman')),
#    and overlap of top 10 and of bottom 10 (count out of 10).
# 3. Table 27: alternative, Spearman rho, top-10 overlap, bottom-10 overlap. Display.
# 4. FIGURE 11 (plot_robustness): scatter of event-weighted (x) vs article-weighted (y) mean tone, one dot per eligible country,
#    45-degree line, the 5 most positive and 5 most negative labelled. Save figures/fig11_robustness.png.

**Figure 11. Mean tone per country weighted by event (horizontal axis) and by number of articles (vertical axis). Each dot is one country; the line marks equal values; the five most positive and five most negative countries are labelled.**

The ranking is [[FILL: highly / moderately]] robust: its Spearman correlation with the alternatives ranges from [[FILL]] to [[FILL]] (Table 27), and [[FILL]] of the bottom ten countries remain in the bottom ten under every alternative. The year-by-year rankings correlate with the full-period ranking at [[FILL]] to [[FILL]], so the ranking describes a lasting pattern rather than a single year. [[FILL: Name the countries that move most and under which choice, if any.]]

## 8. RQ2: Does tone follow seasonal patterns?

### Step 22. Global Tone and Volume by Month

Before looking at countries, Figure 12 shows the global monthly series, to separate long-term movement from any repeating calendar pattern.

In [ ]:
# STEP 22: global monthly series (from global_month_pd, Step 18)

# 1. G_m = global mean tone per month (sum_tone / n_events). events_per_day = n_events / days_available.
# 2. FIGURE 12 (plot_global_monthly): two stacked panels sharing the x-axis (77 months):
#    (top) global mean tone per month with its 2x12 moving average (Eq. 4 applied to G);
#    (bottom) events per available day.
#    Shade December-January bands lightly to help the eye. Save figures/fig12_global_monthly.png.

**Figure 12. Global monthly mean AvgTone (top, with its 2x12 moving-average trend) and events per available day (bottom), April 2013 to August 2019. Shaded bands mark December and January.**

Global tone [[FILL: describe the trend, e.g. drifted downward from X in 2013 to Y in 2019]], and volume [[FILL: describe]]. A trend of this size would be mistaken for seasonality if months were compared directly, which is why every series is detrended before the seasonal index is computed (Eq. 4 and Eq. 5).

### Step 23. Relative Tone, Trend and Anomaly

For each country, monthly tone is first taken relative to the global mean of the same month (Eq. 3), the slow trend is removed with a 2x12 moving average (Eq. 4), and the remainder (Eq. 5) is what seasonality is measured on. Only countries with at least 30 events in **every** month are analysed, so that no monthly mean rests on a handful of events. Figure 13 shows the three stages for one country.

In [ ]:
# STEP 23: anomalies  (uses moving_average_2x12 from gdelt_analysis.py)

# 1. Pivot country_month_pd to a countries x months table of T_{c,m} = sum_tone / n_events (77 columns, months in order).
# 2. Seasonal eligibility: n_events >= MIN_EVENTS_MONTH in every one of the 77 months. Print the number of eligible countries and their % of N_CLEAN events.
#    (This is a filtering step: state the count and % in the text.)
# 3. R = T - G (Eq. 3), broadcast G_m across countries.
# 4. trend = moving_average_2x12(R) along the month axis (weights 1/24, 1/12 x 11, 1/24; NaN for the first and last 6 months).
# 5. A = R - trend (Eq. 5). Keep A only for the 65 months where the trend exists.
# 6. Do the same for the GLOBAL series G itself (G minus its own 2x12 MA) -> global anomalies.
# 7. FIGURE 13 (plot_decomposition_example) for ONE large country (e.g. the highest-volume country): three panels:
#    raw T and G; R with its trend; anomaly A with calendar-month labels. Save figures/fig13_decomposition_example.png.

**Figure 13. How seasonality is isolated, shown for [[FILL: country]]: (a) monthly mean tone of the country and of the world; (b) tone relative to the world (Eq. 3) with its 2x12 trend (Eq. 4); (c) the remaining monthly anomaly (Eq. 5), on which the seasonal index is computed.**

[[FILL]] countries meet the requirement of at least 30 events in every month; together they account for [[FILL]]% of all located events. Figure 13 shows why the two adjustments matter: [[FILL: e.g. the raw series falls with the world's tone, the relative series removes that, and the anomaly keeps only month-to-month movement]].

### Step 24. Seasonal Index and Permutation Test

**Table 28. Seasonal amplitude (Eq. 6), peak and trough month, permutation p-value (Eq. 7) and Benjamini-Hochberg q-value for the countries eligible for seasonality, sorted by amplitude. The 20 largest amplitudes are shown; the summary line covers all countries.**

**Table 29. Global seasonal index by calendar month and its permutation test.**

In [ ]:
# STEP 24: seasonal index, amplitude, significance  (uses seasonal_index, permutation_amplitude_test, benjamini_hochberg)

# 1. seasonal_index(A): for each country, mean of A per calendar month (1..12), minus the mean of those 12 values (so they sum to 0).
# 2. amplitude = max - min of the 12 values; peak_month = argmax; trough_month = argmin.
# 3. permutation_amplitude_test(A, n_perm=N_PERM, seed=SEED):
#    for each permutation, shuffle the calendar-month labels WITHIN each year (keeps year structure), recompute the amplitude;
#    p = (1 + count(perm_amp >= observed)) / (1 + N_PERM)  (Eq. 7). Vectorise across countries with numpy.
# 4. q = benjamini_hochberg(p) across countries; significant = q < ALPHA.
# 5. Table 28: country, events, amplitude, peak month name, trough month name, p, q, significant. Show the top 20 by amplitude.
#    Then print the summary line: X of Y eligible countries (Z%) have significant seasonality at FDR 5%.
# 6. Table 29: the global seasonal index (12 values from the global anomalies), amplitude, p. Display.
# 7. Sensitivity (Design Constraints): repeat the count of significant countries using year-mean subtraction instead of the 2x12 MA. Print both counts side by side.

At the global level, tone [[FILL: shows / does not show]] a significant seasonal pattern (amplitude [[FILL]], p = [[FILL]]; Table 29)[[FILL: , with its brightest month in X and darkest in Y]]. At the country level, [[FILL: k]] of [[FILL: n]] eligible countries ([[FILL]]%) have significant seasonality at a false discovery rate of 5% (Table 28). Among them, the median amplitude is [[FILL]] tone points, compared with a gap of [[FILL]] points between the most positive and most negative countries in Table 26: [[FILL: season moves a country's tone far less than its identity does / season is comparable in size to the differences between countries]]. Peak months [[FILL: cluster in X / are spread across the year]], and trough months [[FILL]]. Using yearly means instead of the 2x12 moving average gives [[FILL]] significant countries instead of [[FILL]], so the conclusion does not depend on the detrending method.

### Step 25. Seasonal Patterns at a Glance

Figure 14 compares the seasonal profiles of many countries on one scale, and Figure 15 shows nine selected countries against the range expected by chance.

In [ ]:
# STEP 25: seasonal figures

# 1. FIGURE 14 (plot_seasonal_heatmap): heatmap, rows = the 30 highest-volume seasonal-eligible countries plus every eligible SEA country,
#    columns = Jan..Dec, cell = seasonal index, diverging colour centred at 0 (same scale for all rows).
#    Order rows by peak month (or by hierarchical clustering of the 12-value profiles; say which).
#    Mark significant countries (q < ALPHA) with an asterisk after the name. Save figures/fig14_seasonal_heatmap.png.
# 2. FIGURE 15 (plot_seasonal_curves): small multiples (3 x 3) for SELECTED_FOR_CURVES:
#    line = seasonal index by calendar month, shaded band = 2.5-97.5% range of the permutation null for that country,
#    horizontal line at 0, same y-limits in all panels, country name + q-value in each panel title.
#    Save figures/fig15_seasonal_curves.png.

**Figure 14. Seasonal index of news tone by calendar month for the 30 most-covered countries and the Southeast Asian countries. Red: darker than the country's own trend in that month; blue: brighter. Asterisks mark countries whose seasonality is significant at a false discovery rate of 5%.**

**Figure 15. Seasonal tone curves for nine selected countries. The grey band is the range expected by chance (2.5th to 97.5th percentile of the permutation distribution); a curve outside the band in some months indicates a seasonal pattern.**

[[FILL: Three sentences from Figures 14 and 15: which months tend to be red or blue across many countries; any group of countries that share a profile; one country whose curve clearly leaves the chance band, and one that stays inside it.]]

## 9. RQ3: Which event types drive each country's tone?

### Step 26. Event Mix by Country

Figure 4 showed that conflict categories are reported far more negatively than cooperative ones. A country can therefore be covered negatively for two different reasons: more of its events are of the kind that is reported negatively everywhere (its *mix*), or its events are reported more negatively than the same kind of event elsewhere (*within* categories). Step 26 builds the country-by-category table needed to separate the two, with a second Spark aggregation.

In [ ]:
# STEP 26: country x CAMEO root sums (second and last big groupBy)

# 1. country_root_df = events_df.groupBy('country_fips', 'event_root_code').agg(n = count, sum_tone = sum(avg_tone)).
# 2. country_root_pd = toPandas(); keep eligible countries; pivot to shares s_ck and mean tones t_ck (20 roots).
#    Missing combinations: share 0; tone undefined -> the decomposition treats 0-share cells as contributing 0 (comment why).
# 3. FIGURE 16 (plot_event_mix): 100% stacked horizontal bars of QuadClass shares (4 colours, consistent with Figure 4)
#    for the 10 most positive and 10 most negative countries, global bar on top. Save figures/fig16_event_mix.png.

**Figure 16. Share of each country's events in the four QuadClass categories, for the ten most positively and the ten most negatively covered countries. The top bar shows the global mix.**

Material conflict makes up [[FILL]]% of events in the ten most negatively covered countries on average, against [[FILL]]% in the ten most positively covered countries and [[FILL]]% worldwide (Figure 16). Event mix therefore differs strongly between the two ends of the ranking; Step 27 measures how much of the tone gap it explains.

### Step 27. Decomposing Each Country's Tone Gap

**Table 30. Decomposition (Eq. 8) of the gap between each country's mean tone and the global mean into an event-mix part and a within-category part, with the CAMEO category that contributes most, for the ten most positive and ten most negative countries.**

In [ ]:
# STEP 27: Kitagawa decomposition  (uses tone_decomposition from gdelt_analysis.py)

# 1. tone_decomposition(country_root_pd, global_root_pd): for each country and root k:
#    mix_k = (s_ck - s_gk) * (t_gk - global_mean);  within_k = s_ck * (t_ck - t_gk)
#    gap = country mean - global mean; mix = sum_k mix_k; within = sum_k within_k.
# 2. ASSERT for every country: abs(gap - (mix + within)) < 1e-9. Print "decomposition adds up for all N countries".
# 3. For each country: the root with the most negative (and most positive) total contribution (mix_k + within_k), with its name.
# 4. Table 30: country, gap, mix, within, % of gap explained by mix, biggest negative root, biggest positive root. Top 10 + bottom 10. Display.
# 5. Summary print: among the bottom 15 countries, in how many is mix the larger part of the gap? Among the top 15?
# 6. FIGURE 17 (plot_decomposition): for the 15 most negative and 15 most positive countries, horizontal bars split into mix (one colour)
#    and within (another colour) around 0, with a black dot at the total gap. Save figures/fig17_decomposition.png.

**Figure 17. Why each country's tone differs from the global mean: the part due to the kind of events that happen there (mix) and the part due to how events of the same kind are reported (within). Dots mark the total gap.**

The two parts add up to the total gap for every country (largest difference [[FILL]], numerical rounding only). Among the 15 most negatively covered countries, event mix accounts for the larger share of the gap in [[FILL]] countries and within-category tone in [[FILL]] (Table 30, Figure 17). The category that most often contributes the largest negative amount is [[FILL: e.g. 19 Fight]], followed by [[FILL]]. [[FILL: Name any country with a large negative within part, i.e. events reported more negatively than the same events elsewhere, and any positive country whose advantage is mostly within.]] The distinction matters for anyone who wants to change a country's coverage: a gap driven by mix reflects what happens there, while a gap driven by within-category tone reflects how it is reported, which communication can address.

## 10. RQ4: How is the Philippines covered compared with its Southeast Asian neighbours?

### Step 28. The Philippines among Its Neighbours

**Table 31. Southeast Asian countries: number of events, pooled mean tone with 95% interval, rank among all eligible countries, share of conflict events (QuadClass 3 and 4) and seasonal amplitude.**

In [ ]:
# STEP 28: SEA comparison (all from tables already computed: no new Spark job)

# 1. sea_pd = country_summary_pd filtered to SEA_FIPS; add rank among ALL eligible countries (from Step 19),
#    conflict share (QuadClass 3+4, from a QuadClass pivot of Step 26 or a small groupBy), seasonal amplitude and q (Step 24, if eligible; else "not eligible").
# 2. SEA aggregate excluding the Philippines: pooled over the other 10 (sum of sums), so big countries weigh more; say so.
# 3. Table 31 sorted by mean tone. Highlight the RP row (bold via pandas Styler, or a marker column). Display.
# 4. FIGURE 18 (plot_sea_panel): two panels:
#    (a) dot plot with CIs for the 11 countries (RP highlighted with the accent colour, global mean dashed line);
#    (b) plotly or matplotlib map zoomed to Southeast Asia coloured on the SAME scale as Figure 10.
#    Save figures/fig18_sea.png (and the map as HTML if plotly).

**Figure 18. News tone of the 11 Southeast Asian countries: (a) pooled mean tone with 95% interval, the Philippines highlighted; (b) map on the same colour scale as Figure 10.**

The Philippines has a pooled mean tone of [[FILL]] (95% interval [[FILL]] to [[FILL]]), ranking [[FILL]] of 11 in Southeast Asia and [[FILL]] of [[FILL]] countries worldwide (Table 31, Figure 18). It is the [[FILL: most / second most]] covered country in the region with [[FILL]] events, and [[FILL]]% of its events are conflict events, against [[FILL]]% for the other Southeast Asian countries combined. [[FILL: Name the most positive and most negative neighbour with values.]]

### Step 29. The Philippines over Time

Step 29 follows the Philippines' monthly tone relative to the world (Eq. 3) against the combined tone of its ten neighbours, with three dated national events marked for reference: the landfall of Typhoon Haiyan (Yolanda) on 8 November 2013 (National Disaster Risk Reduction and Management Council [NDRRMC], 2014), the presidential inauguration of 30 June 2016 (Dizon, 2016), and the siege of Marawi from 23 May to 23 October 2017 (Placido, 2017). The periods before and after July 2016 are then compared with Eq. 9. The comparison is descriptive: it shows whether coverage of the Philippines changed more than coverage of its neighbours, not why.

**Table 32. Mean relative tone of the Philippines and of the other Southeast Asian countries before and after July 2016, and the difference of the changes (Eq. 9) with its 95% month-bootstrap interval.**

In [ ]:
# STEP 29: PH monthly series and before/after comparison  (uses period_change from gdelt_analysis.py)

# 1. Monthly relative tone R (Eq. 3) for RP and for the pooled other-SEA group, plus their 3-month rolling means.
# 2. PH_EVENTS (in gdelt_lookups.py): a short list of dated events WITH a source each (cite them in References), e.g.
#    Typhoon Haiyan landfall 2013-11-08; presidential inauguration 2016-06-30; Marawi siege 2017-05-23 to 2017-10-23.
#    (Verify every date against the cited source before using it.)
# 3. FIGURE 19 (plot_ph_timeline): RP line (accent colour) vs other-SEA line (grey), thin monthly + thick rolling,
#    vertical lines / shaded spans for PH_EVENTS with short labels, horizontal line at 0 (= global mean). Save figures/fig19_ph_timeline.png.
# 4. period_change(..., split_month=PH_SPLIT_MONTH): mean R before and after for RP and for other SEA; Delta (Eq. 9);
#    95% interval by bootstrapping months within each period (seed SEED). Table 32. Display.
# 5. Event-type view of the change: RP shares of QuadClass 3+4 before vs after (small table or printed lines).

**Figure 19. Monthly news tone of the Philippines (accent colour) and of the other Southeast Asian countries (grey), relative to the global mean of each month, with three national events marked.**

[[FILL: Two sentences describing Figure 19: whether the Philippines is usually above or below its neighbours, and which movements coincide with the marked events (use "coincides with", never "caused").]] After July 2016, the Philippines' relative tone changed by [[FILL]] points, against [[FILL]] for its neighbours, a difference of Δ = [[FILL]] (95% interval [[FILL]] to [[FILL]]; Table 32), which [[FILL: excludes / includes]] zero. Over the same periods, the share of conflict events in Philippine coverage moved from [[FILL]]% to [[FILL]]%.

### Step 30. Season, Drivers and Regions of Philippine Coverage

**Table 33. The ten first-level regions of the Philippines with the most located events: number of events and mean tone.**

In [ ]:
# STEP 30: PH deep-dive (three outputs)

# 1. FIGURE 20 (plot_ph_seasonal): seasonal index of RP vs the mean seasonal index of the other eligible SEA countries,
#    with RP's permutation null band (from Step 24). Shade June to November (typhoon season) lightly, ONLY as a reference;
#    the text must say whether the data shows darker tone there or not.
#    If RP is not seasonal-eligible, say so and use the next eligible grouping. Save figures/fig20_ph_seasonal.png.
# 2. FIGURE 21 (plot_ph_drivers): decomposition (Eq. 8) with the OTHER-SEA pool as the reference instead of the world:
#    for RP, per CAMEO root, bar of mix_k + within_k (sorted), split by colour into mix and within. Title-free; caption carries it.
#    Assert the parts add to RP's gap vs other SEA. Save figures/fig21_ph_drivers.png.
# 3. Table 33: from events_df filtered to country_fips == 'RP' and geo_type in (4, 5) (sub-national locations):
#    groupBy adm1 -> events, mean tone; region name = most common text before the first comma of geo_fullname within that adm1
#    (comment the heuristic). Top 10 by events. Display.

**Figure 20. Seasonal tone index of the Philippines compared with the average index of the other eligible Southeast Asian countries. The grey band is the range expected by chance for the Philippines; light shading marks June to November, the main typhoon season.**

**Figure 21. Contribution of each CAMEO event category to the gap between the Philippines' mean tone and that of its Southeast Asian neighbours, split into event mix and within-category tone.**

Philippine news tone [[FILL: shows / does not show]] significant seasonality (q = [[FILL]]), and Figure 20 shows that tone during June to November is [[FILL: lower than / similar to / higher than]] in the rest of the year. Relative to its neighbours, the Philippines' tone gap of [[FILL]] points comes mainly from [[FILL: mix / within]]; the categories that pull it down most are [[FILL]] and [[FILL]], while [[FILL]] pulls it up (Figure 21). Within the country, [[FILL: region]] carries the largest share of located coverage ([[FILL]] events, mean tone [[FILL]]), and [[FILL: region]] has the most negative tone among the ten regions in Table 33.

## 11. Results Summary and Conclusion

### 11.1 Results by Research Question

**Table 34. Answer to each research question, the key numbers behind it, and the figures and tables that show it.**

| RQ | Answer | Key numbers | Evidence |
|---|---|---|---|
| RQ1 | [[FILL: one sentence naming the most positive and most negative countries]] | [[FILL: their mean tones and the global mean]] | Table 26, Figures 9 to 10 |
| RQ2 | [[FILL: one sentence on global and country seasonality]] | [[FILL: k of n countries; median amplitude]] | Tables 28 to 29, Figures 14 to 15 |
| RQ3 | [[FILL: one sentence on mix versus within]] | [[FILL: count of countries where mix dominates; top category]] | Table 30, Figure 17 |
| RQ4 | [[FILL: one sentence on the Philippines' position and change]] | [[FILL: rank, mean tone, Δ]] | Tables 31 to 33, Figures 18 to 21 |

**RQ1: Ranking.** Global news covers [[FILL: most positive countries]] most positively and [[FILL: most negative countries]] most negatively (Table 26). The ranking is not an artefact of how tone is averaged: it holds when events are weighted by articles, restricted to lead-paragraph events, summarised by the median, or computed one year at a time (Spearman correlations of [[FILL]] to [[FILL]], Table 27). [[FILL: One further insight from Figure 10, e.g. a regional pattern.]]

**RQ2: Seasonality.** [[FILL: Global seasonality result.]] At the country level, [[FILL]] of [[FILL]] countries show seasonality beyond chance (Table 28). The typical seasonal swing of [[FILL]] tone points is [[FILL: much smaller than / comparable to]] the [[FILL]]-point gap between the most and least favourably covered countries, so *where* an event happens matters [[FILL: far more than / about as much as]] *when* it is reported.

**RQ3: Drivers.** For the most negatively covered countries, [[FILL: event mix / within-category tone]] explains most of the gap from the global mean (Table 30, Figure 17), with [[FILL: category]] the most frequent negative contributor. [[FILL: One insight, e.g. countries whose events are reported more negatively than the same events elsewhere.]]

**RQ4: The Philippines.** The Philippines ranks [[FILL]] of 11 in Southeast Asia and [[FILL]] worldwide (Table 31). [[FILL: Its relative tone changed by Δ = X after July 2016 compared with its neighbours (Table 32)]], and its gap from its neighbours comes mainly from [[FILL: categories]] (Figure 21). [[FILL: One sentence on seasonality from Figure 20.]]

### 11.2 Conclusion

Global news portrays [[FILL: three most positive countries]] most positively and [[FILL: three most negative countries]] most negatively, with mean tones of [[FILL]] to [[FILL]] against a global mean of [[FILL]], and tone follows a significant seasonal pattern in [[FILL: k]] of [[FILL: n]] countries but [[FILL: does / does not]] follow one at the global level. The differences between countries are [[FILL: mostly due to the kind of events that happen in them / mostly due to how their events are reported]], and seasonal swings are [[FILL: small / large]] compared with those differences. The Philippines is covered [[FILL: more / less]] positively than most of its Southeast Asian neighbours, ranking [[FILL]] of 11, with [[FILL: main driver from Figure 21]] pulling its tone [[FILL: down / up]].

### 11.3 Significance and Actionable Insights

The results give a measurable, repeatable view of national image that can be updated each month from public data with the same Spark pipeline. Five groups can act on them:

1. **Tourism boards and investment promotion agencies**, such as the Philippine Department of Tourism, can adopt the country's relative tone (Eq. 3) as a monthly key indicator, benchmarked against its neighbours (Table 31). A campaign's effect can then be judged by whether relative tone rises above its pre-campaign level, rather than by impressions alone.
2. **Government communications offices** can read Figure 21 as a to-do list. Where a country's gap from its peers comes from *within-category* tone, the same kind of event is reported more negatively than elsewhere, which proactive briefings, data releases and access for foreign press can address. Where it comes from *mix*, communication cannot change the coverage until the events themselves change.
3. **Planners of national announcements** can use the seasonal index (Figures 14 and 20) to avoid launching positive campaigns in the months when coverage is usually darkest[[FILL: , which for the Philippines are X and Y]], and to prepare messaging before them.
4. **Investors and country-risk analysts** (beyond the tourism context) can track relative tone as a cheap, early signal of shifts in international perception, since GDELT updates daily and the pipeline processes over six years of data in [[FILL]] minutes.
5. **Newsrooms, media monitors and researchers** (beyond the study's context) can use the within-category component of Eq. 8 as an indicator of framing differences: two countries experiencing the same kind of event but receiving systematically different tone is a measurable question for editorial review and for research on media bias.

## 12. Limitations, Recommendations and Future Work

### 12.1 Limitations

**Table 35. Limitations of the study, their likely effect on the results, and how each was mitigated or could be.**

| # | Limitation | Likely effect on the results | Mitigation in this study / possible fix |
|---|---|---|---|
| 1 | Automated event coding misclassifies and duplicates some events (Wang et al., 2016) | Noise in event counts and in the share of each category | Large samples; intervals that respect dependence within months; results reported as relative comparisons |
| 2 | GDELT's coverage of sources differs from other news archives (Kwak & An, 2016) and changes over time | Absolute tone and volume reflect GDELT's source mix | Tone measured relative to the global mean of each month (Eq. 3) and detrended |
| 3 | Tone is a dictionary-based word count, partly on machine-translated text | Sarcasm, context and quotations are not understood | Only differences between countries and months are interpreted, never absolute values |
| 4 | Events without a location ([[FILL]]% of events) are removed | Coverage that names no place is not attributed to any country | Investigated before removal (Step 12); tone of removed events compared |
| 5 | Each event's tone is fixed in the daily file in which it was added | Later coverage of the same event is not reflected | Stated as part of the definition of tone |
| 6 | Volume is not normalised by population or media market size | Large, much-covered countries dominate pooled groups such as "other Southeast Asia" | Rankings use means, not totals; pooled groups are labelled as event-weighted |
| 7 | FIPS 10-4 codes do not map one-to-one to today's ISO countries, and some codes denote regions or oceans | A few codes cannot be placed on the map | Unmapped codes listed with their events (Table 22) |
| 8 | Four days have no file, and the data end in August 2019 | Results describe 2013 to 2019 and may not hold today | Volume expressed per available day; extension proposed below |
| 9 | The design is descriptive | The before/after comparison for the Philippines (Eq. 9) cannot establish cause | Stated wherever the comparison is reported |

### 12.2 Recommendations and Future Work

1. **Extend the period to the present with GDELT 2.0.** GDELT 2.0 updates every 15 minutes from 2015 and adds a mentions table that records every article citing an event. Re-running the pipeline on it would show whether the 2013 to 2019 ranking still holds after events such as the COVID-19 pandemic, and would allow tone to be followed across all later coverage of an event rather than fixed when it was added (Limitations 5 and 8).
2. **Separate who reports from who is reported on.** The domain of each article's URL indicates the country of the outlet. Splitting tone by reporting country would show whether a country is covered more negatively by foreign media than by its own, which is the part of its image a tourism board most needs to understand (Limitation 2).
3. **Validate the tone measure.** A random sample of articles per country could be scored with a modern multilingual sentiment model, or by human coders, and compared with `AvgTone`. A strong agreement would support the dictionary measure; a weak one would show which countries or languages it misjudges (Limitation 3).
4. **Normalise and group countries.** Dividing event volume by population and by the size of each country's media market, and grouping countries by hemisphere and climate, would test whether seasonal peaks follow local seasons, holidays or the northern-hemisphere news calendar (Limitation 6).
5. **Move from description to causal evidence for the Philippines.** An event-study design around specific dates, with a synthetic control built from the neighbouring countries, would estimate how much specific events changed the country's coverage, which the descriptive comparison in Step 29 cannot do (Limitation 9).

## References

Alis, C. (n.d.). [[FILL: titles of the course notebooks used, in the same form as in Lab 1, e.g. "Introduction to Apache Spark; Spark SQL and DataFrames"]] [Course notebooks]. BDCC 3301.

Anholt, S. (2007). *Competitive identity: The new brand management for nations, cities and regions*. Palgrave Macmillan.

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. In *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). ACM. https://doi.org/10.1145/2723372.2742797

Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: A practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B (Methodological), 57*(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x

Dizon, D. (2016, June 30). Duterte: Change must start in us. *ABS-CBN News*. https://abs-cbn.com/nation/06/30/16/duterte-change-must-start-in-us

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.

GDELT Project. (2013). *GDELT data format codebook V1.03*. http://data.gdeltproject.org/documentation/GDELT-Data_Format_Codebook.pdf

GDELT Project. (n.d.). *Data*. Retrieved November 15, 2026, from https://www.gdeltproject.org/data.html

Good, P. (2005). *Permutation, parametric, and bootstrap tests of hypotheses* (3rd ed.). Springer.

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3rd ed.). OTexts. https://otexts.com/fpp3/

Kitagawa, E. M. (1955). Components of a difference between two rates. *Journal of the American Statistical Association, 50*(272), 1168–1194. https://doi.org/10.1080/01621459.1955.10501299

Kwak, H., & An, J. (2016). Two tales of the world: Comparison of widely used world news datasets GDELT and EventRegistry. *Proceedings of the International AAAI Conference on Web and Social Media, 10*(1), 619–622. https://doi.org/10.1609/icwsm.v10i1.14763

Leetaru, K., & Schrodt, P. A. (2013, April). *GDELT: Global data on events, location, and tone, 1979–2012* [Paper presentation]. International Studies Association Annual Convention, San Francisco, CA, United States. http://data.gdeltproject.org/documentation/ISA.2013.GDELT.pdf

National Disaster Risk Reduction and Management Council. (2014). *Final report on the effects of Typhoon Yolanda (Haiyan)*. Republic of the Philippines.

Placido, D. (2017, October 23). Gov't sets sights on rebuilding Marawi as siege ends. *ABS-CBN News*. https://news.abs-cbn.com/news/10/23/17/govt-sets-sights-on-rebuilding-marawi-as-siege-ends

Schrodt, P. A. (2012). *CAMEO: Conflict and Mediation Event Observations event and actor codebook* (Version 1.1b3). Pennsylvania State University. http://data.gdeltproject.org/documentation/CAMEO.Manual.1.1b3.pdf

Soroka, S., & McAdams, S. (2015). News, politics, and negativity. *Political Communication, 32*(1), 1–22. https://doi.org/10.1080/10584609.2014.881942

van Rossum, G., Warsaw, B., & Coghlan, N. (2001). *PEP 8 – Style guide for Python code*. Python Software Foundation. https://peps.python.org/pep-0008/

Wang, W., Kennedy, R., Lazer, D., & Ramakrishnan, N. (2016). Growing pains for global monitoring of societal events. *Science, 353*(6307), 1502–1503. https://doi.org/10.1126/science.aaf6758

Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S., & Stoica, I. (2016). Apache Spark: A unified engine for big data processing. *Communications of the ACM, 59*(11), 56–65. https://doi.org/10.1145/2934664

*Data acknowledgement.* This report uses the GDELT 1.0 Event Database, an open dataset published by the GDELT Project, read in place from the shared Jojie folder and cited as the project requests. The raw data are not redistributed with this report.

# SUBMISSION CHECKLIST (DELETE THIS CELL BEFORE SUBMITTING)

The final report is due by **end of day, 16 November 2026**.

**A. `du -sch` screenshot**
- [ ] Open a new Jojie terminal and hide the username first: `export PS1='$ '` then `clear`.
- [ ] Run `cd /mnt/data/public/gdeltv1/events && du -sch *.export.CSV.zip | tail -3`. These are exactly the 2,335 files the notebook reads; the last line shows the total (24G).
- [ ] Screenshot only the terminal area (no browser bar or tab titles) and save it as `du_screenshot.png` next to the notebook.

**B. Fill every gap**
- [ ] Ctrl+F for `[[` returns 0 results. Every filled number matches a printed output.
- [ ] Re-read each interpretive sentence against its figure or table, and rewrite it if the output says otherwise.
- [ ] Check every reference once more, and fill in the Alis (n.d.) titles to match Lab 1.

**C. Runs top to bottom**
- [ ] Kernel > Restart Kernel and Run All Cells: no errors, no warnings showing paths.
- [ ] If time allows, run once with an empty `scratch/` and record the runtime in Section 3.5.

**D. Anonymity**
- [ ] Delete the READ ME cell and this checklist.
- [ ] `grep -i -c "<your username>" main.ipynb main.html gdelt_*.py` prints 0 for every file; repeat for your real name and student number.
- [ ] `grep -c "/home/" main.html` prints 0.
- [ ] The folder and file names are neutral; no figure contains a name.

**E. Export and package**
- [ ] File > Save and Export Notebook As > HTML; open the HTML and check that figures show, maps are interactive and the Table of Contents links work.
- [ ] Submit `main.ipynb`, `main.html`, `du_screenshot.png`, the four `gdelt_*.py` files and `figures/`. Do not submit `scratch/` or any data files.